# Smoke test

Proves the remote kernel actually works before any research time is spent on it.
Run top to bottom. Every cell should print something and none should raise.

If cell 1 prints your *laptop's* hostname, you are on the local kernel — go back and
select the remote one.

In [ ]:
# 1. Where am I, and is there a GPU?
import os, socket, subprocess, sys

print("hostname :", socket.gethostname())
print("python   :", sys.version.split()[0])
print("WORKSPACE:", os.environ.get("WORKSPACE", "(unset — local kernel?)"))
print("HF_HOME  :", os.environ.get("HF_HOME", "(unset)"))
print()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)

In [ ]:
# 2. torch sees CUDA
import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "no CUDA — wrong image or a bad host; just down and retry"
print(torch.cuda.get_device_name(0))

# A real allocation, not just a capability check.
x = torch.randn(4096, 4096, device="cuda")
print("matmul ok:", (x @ x).sum().item() is not None)
del x; torch.cuda.empty_cache()

In [ ]:
# 3. Project config resolves
from nandaproj import config

cfg = config.get_model_config()
config.ensure_dirs()
print(cfg)
print("device:", config.get_device())
print("cache :", config.HF_CACHE)

In [ ]:
# 4. TransformerLens: load, forward pass, cached activations
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained(cfg.name, device="cuda")
print(f"{cfg.name}: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, "
      f"d_model={model.cfg.d_model}")

prompt = "When John and Mary went to the store, John gave a drink to"
logits, cache = model.run_with_cache(prompt)
print("logits:", tuple(logits.shape))
print("top prediction:", repr(model.to_string(logits[0, -1].argmax())))

pattern = cache["pattern", 0]
print("layer-0 attention pattern:", tuple(pattern.shape))

In [ ]:
# 5. Plotting round-trips through the tunnel
from nandaproj.viz import imshow

imshow(pattern[0], title="L0H0 attention", xaxis="key pos", yaxis="query pos")

In [ ]:
# 6. nnsight imports and can trace
import nnsight
from nnsight import LanguageModel

print("nnsight", nnsight.__version__)
lm = LanguageModel("openai-community/gpt2", device_map="cuda")
with lm.trace("The Eiffel Tower is in"):
    hidden = lm.transformer.h[6].output[0].save()
print("traced hidden state:", tuple(hidden.shape))

In [ ]:
# 7. SAELens loads a pretrained SAE
# gpt2-small-res-jb is ungated, so this works without an HF token.
# Swap to a Gemma Scope release once HF_TOKEN is set and you move to Gemma-2-2B.
from sae_lens import SAE

sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    device="cuda",
)
sae = sae[0] if isinstance(sae, tuple) else sae
print("SAE d_in:", sae.cfg.d_in, "| d_sae:", sae.cfg.d_sae)

acts = cache["blocks.7.hook_resid_pre"]
features = sae.encode(acts)
print("feature acts:", tuple(features.shape),
      "| L0 (avg live features):", (features > 0).float().sum(-1).mean().item())

If all seven cells ran, the environment is green: remote GPU, TransformerLens, nnsight,
SAELens, plotting, and the shared kernel.

**Now run `just down`.**